In [1]:
# 00_understanding_states.py
"""
FlagQuantum Tutorial - Lesson 0: Understanding State Tensor Structure
Goal: Thoroughly understand how FlagQuantum represents quantum states
"""
import flagquantum as fq


def tutorial_00_basic_tensor_structure():
    """Understand the basic structure of state tensors"""
    print("=" * 70)
    print("Lesson 0.0: Basic Structure of State Tensors")
    print("=" * 70)

    # Create devices of different sizes
    configs = [
        (1, "Single qubit"),
        (2, "Two qubits"),
        (3, "Three qubits"),
    ]

    for n_wires, name in configs:
        qdev = fq.DistributedQuantumDevice(n_wires=n_wires, bsz=1, device="cpu")
        print(f"\n{name}: n_wires={n_wires}")
        print(f"  State tensor shape: {qdev.states.shape}")
        print(f"  Interpretation: {qdev.states.shape} = [batch_size, {', '.join([f'q{i}' for i in range(n_wires)])}, real/imag]")

In [2]:
tutorial_00_basic_tensor_structure()

Lesson 0.0: Basic Structure of State Tensors

Single qubit: n_wires=1
  State tensor shape: torch.Size([1, 2, 2])
  Interpretation: torch.Size([1, 2, 2]) = [batch_size, q0, real/imag]

Two qubits: n_wires=2
  State tensor shape: torch.Size([1, 2, 2, 2])
  Interpretation: torch.Size([1, 2, 2, 2]) = [batch_size, q0, q1, real/imag]

Three qubits: n_wires=3
  State tensor shape: torch.Size([1, 2, 2, 2, 2])
  Interpretation: torch.Size([1, 2, 2, 2, 2]) = [batch_size, q0, q1, q2, real/imag]


In [3]:
import torch

"""
FlagQuantum Endianness Quick Reference Card
===========================================
Endianness type: Big-endian ✅
Shape format: [batch, q0, q1, ..., q_{N-1}, 2]
Qubit order: q0 = MSB (Most Significant Bit), q_{N-1} = LSB

Basis state examples (n=3):
  |000⟩ → [batch, 0,0,0, :]
  |001⟩ → [batch, 0,0,1, :]   (q2=1)
  |010⟩ → [batch, 0,1,0, :]   (q1=1)
  |100⟩ → [batch, 1,0,0, :]   (q0=1)
"""


def test_endianness():
    """Verify FlagQuantum's endianness by manually computing probabilities"""
    print("\n" + "=" * 70)
    print("Lesson 0.1: Determine Qubit Ordering (Big-endian vs Little-endian)")
    print("=" * 70)
    # 1. Create a 2-qubit device, initial state |00>
    qdev = fq.DistributedQuantumDevice(n_wires=2, bsz=1, device="cpu")

    # Helper function: compute probability distribution from states tensor
    def get_probs(device):
        # Assume states shape is [batch, q0, q1, ..., 2]
        real = device.states[..., 0]
        imag = device.states[..., 1]
        state_vector = torch.complex(real, imag)
        # Compute squared magnitude to get probabilities, flatten to 1D for viewing
        probs = (state_vector.abs()) ** 2
        return probs.flatten()

    print("=== FlagQuantum Endianness Test ===")
    print(f"states original shape: {qdev.states.shape}")

    # 2. Check initial state probabilities
    probs_initial = get_probs(qdev)
    print(f"\nInitial state probabilities (should be |00> = 1.0):\n  {probs_initial}")

    # 3. Apply X gate to the first qubit (wires=0)
    qdev.x(wires=0)
    probs_after_x = get_probs(qdev)
    print(f"\nProbabilities after applying X gate to q0:\n  {probs_after_x}")

    # 4. Determine endianness
    # Little-endian expectation: index 1 (binary 01) has probability 1
    # Big-endian expectation: index 2 (binary 10) has probability 1
    if probs_after_x[1] > 0.99:
        print("\n✅ Conclusion: Little-endian")
        print("   Interpretation: q0 is LSB, |00> -> X(q0) -> |01>")
    elif probs_after_x[2] > 0.99:
        print("\n✅ Conclusion: Big-endian")
        print("   Interpretation: q0 is MSB, |00> -> X(q0) -> |10>")
    else:
        print("\n⚠️ Cannot determine, please check if gate operations are working or if qubit count is correct")

    # 5. Optional: Test CNOT for further verification
    print("\n--- Advanced verification: CNOT gate ---")
    # Reset device or create a new one (simply create a new one here)
    qdev2 = fq.DistributedQuantumDevice(n_wires=2, bsz=1, device="cpu")
    # Prepare |01> state (in Little-endian: q0=1)
    qdev2.x(wires=0)
    print(f"Probabilities of prepared |01> state: {get_probs(qdev2)}")

    # CNOT: control=q0, target=q1
    qdev2.cx(wires=[0, 1])
    probs_cnot = get_probs(qdev2)
    print(f"After CNOT(q0->q1): {probs_cnot}")

    if probs_cnot[3] > 0.99:  # index 3 corresponds to |11> (binary 11)
        print("CNOT verification passed: control-target interaction behaves as expected")

if __name__ == "__main__":
    test_endianness()


Lesson 0.1: Determine Qubit Ordering (Big-endian vs Little-endian)
=== FlagQuantum Endianness Test ===
states original shape: torch.Size([1, 2, 2, 2])

Initial state probabilities (should be |00> = 1.0):
  tensor([1., 0., 0., 0.])

Probabilities after applying X gate to q0:
  tensor([0., 0., 1., 0.])

✅ Conclusion: Big-endian
   Interpretation: q0 is MSB, |00> -> X(q0) -> |10>

--- Advanced verification: CNOT gate ---
Probabilities of prepared |01> state: tensor([0., 0., 1., 0.])
After CNOT(q0->q1): tensor([0., 0., 0., 1.])
CNOT verification passed: control-target interaction behaves as expected


In [4]:
def tutorial_02_read_single_qubit_state():
    """Read and understand the state vector"""
    print("\n" + "=" * 70)
    print("Lesson 0.2: Reading and Understanding the State Vector")
    print("=" * 70)

    qdev = fq.DistributedQuantumDevice(n_wires=2, bsz=1, device="cpu")

    print("\n1. Basis state: |00⟩")
    states = torch.view_as_complex(qdev.states)
    print(f"   Tensor shape: {states.shape}")
    print(f"   State: {states}")
    print(f"   Flattened: {states.flatten()}")
    print("   Meaning: states[0,0] = 1.0 → 100% probability in |00⟩")

    print("\n2. Apply X gate on qubit 0: |10⟩")
    qdev.reset_states()
    fq.X(wires=[0])(qdev)
    states = torch.view_as_complex(qdev.states)
    print(f"   State: {states.flatten()}")
    nonzero = torch.where(states.abs() > 0.5)
    print(f"   Non-zero indices: {nonzero}")

    print("\n3. Apply X gate on qubit 1: |01⟩")
    qdev.reset_states()
    fq.X(wires=[1])(qdev)
    states = torch.view_as_complex(qdev.states)
    print(f"   State: {states.flatten()}")
    nonzero = torch.where(states.abs() > 0.5)
    print(f"   Non-zero indices: {nonzero}")

    print("\n4. Apply X gate on both qubits: |11⟩")
    qdev.reset_states()
    fq.X(wires=[0])(qdev)
    fq.X(wires=[1])(qdev)
    states = torch.view_as_complex(qdev.states)
    print(f"   State: {states.flatten()}")

In [5]:
tutorial_02_read_single_qubit_state()


Lesson 0.2: Reading and Understanding the State Vector

1. Basis state: |00⟩
   Tensor shape: torch.Size([1, 2, 2])
   State: tensor([[[1.+0.j, 0.+0.j],
         [0.+0.j, 0.+0.j]]])
   Flattened: tensor([1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j])
   Meaning: states[0,0] = 1.0 → 100% probability in |00⟩

2. Apply X gate on qubit 0: |10⟩
   State: tensor([0.+0.j, 0.+0.j, 1.+0.j, 0.+0.j])
   Non-zero indices: (tensor([0]), tensor([1]), tensor([0]))

3. Apply X gate on qubit 1: |01⟩
   State: tensor([0.+0.j, 1.+0.j, 0.+0.j, 0.+0.j])
   Non-zero indices: (tensor([0]), tensor([0]), tensor([1]))

4. Apply X gate on both qubits: |11⟩
   State: tensor([0.+0.j, 0.+0.j, 0.+0.j, 1.+0.j])


In [6]:
def tutorial_03_amplitude_to_probability():
    """From amplitude to probability"""
    print("\n" + "=" * 70)
    print("Lesson 0.3: From Amplitude to Probability")
    print("=" * 70)

    qdev = fq.DistributedQuantumDevice(n_wires=2, bsz=1, device="cpu")

    # Create superposition state
    fq.H(wires=[0])(qdev)
    fq.H(wires=[1])(qdev)

    states = torch.view_as_complex(qdev.states)
    print(f"Superposition state amplitudes: {states.flatten()}")
    print(f"All amplitudes are complex numbers: {states.dtype}")

    # Calculate probabilities
    probs = torch.abs(states) ** 2
    print(f"\nProbability distribution: {probs.flatten()}")
    print(f"Sum of probabilities: {probs.sum().item():.1f} (should be 1.0)")

    # Verify normalization
    print("\nNormalization verification:")
    for i in range(4):
        binary = format(i, '02b')
        prob = probs.flatten()[i].item()
        amp = states.flatten()[i]
        print(f"  |{binary}⟩: amplitude={amp:.3f}, probability={prob:.3f}")

In [7]:
tutorial_03_amplitude_to_probability()


Lesson 0.3: From Amplitude to Probability
Superposition state amplitudes: tensor([0.5000+0.j, 0.5000+0.j, 0.5000+0.j, 0.5000+0.j])
All amplitudes are complex numbers: torch.complex64

Probability distribution: tensor([0.2500, 0.2500, 0.2500, 0.2500])
Sum of probabilities: 1.0 (should be 1.0)

Normalization verification:
  |00⟩: amplitude=0.500+0.000j, probability=0.250
  |01⟩: amplitude=0.500+0.000j, probability=0.250
  |10⟩: amplitude=0.500+0.000j, probability=0.250
  |11⟩: amplitude=0.500+0.000j, probability=0.250


In [8]:
def tutorial_04_batch_processing():
    """Batch processing of states"""
    print("\n" + "=" * 70)
    print("Lesson 0.4: Batch Processing of States")
    print("=" * 70)

    batch_size = 3
    n_wires = 2
    qdev = fq.DistributedQuantumDevice(n_wires=n_wires, bsz=batch_size, device="cpu")

    print(f"Batch size: {batch_size}")
    print(f"State tensor shape: {qdev._states.shape}")
    print(f"Interpretation: [{batch_size}, {n_wires} qubit dimensions, real/imag]")

    # Each batch is processed independently
    print("\nSetting different initial states for each batch:")
    for i in range(batch_size):
        qdev.reset_states()  # Reset all batches
        # Here we can only set globally, cannot set each batch individually
        # Batch independence is achieved through parameterized gates

    # Using batch parameters
    thetas = torch.tensor([[0.5], [1.0], [1.5]])  # Different angle for each batch
    print(f"Batch parameters: {thetas.squeeze()}")

    fq.RY(wires=[0])(qdev, params=thetas)

    states = torch.view_as_complex(qdev.states)
    print("\nStates for each batch:")
    for i in range(batch_size):
        probs = torch.abs(states[i]) ** 2
        print(f"  Batch {i}: probability |0⟩={probs[0,0].item():.3f}, |1⟩={probs[1,0].item():.3f}")

In [9]:
tutorial_04_batch_processing()


Lesson 0.4: Batch Processing of States
Batch size: 3
State tensor shape: torch.Size([3, 2, 2, 2])
Interpretation: [3, 2 qubit dimensions, real/imag]

Setting different initial states for each batch:
Batch parameters: tensor([0.5000, 1.0000, 1.5000])

States for each batch:
  Batch 0: probability |0⟩=0.939, |1⟩=0.061
  Batch 1: probability |0⟩=0.770, |1⟩=0.230
  Batch 2: probability |0⟩=0.535, |1⟩=0.465


In [10]:
def tutorial_05_complex_numbers():
    """Understanding complex amplitudes"""
    print("\n" + "=" * 70)
    print("Lesson 0.5: Understanding Complex Amplitudes")
    print("=" * 70)

    qdev = fq.DistributedQuantumDevice(n_wires=1, bsz=1, device="cpu")

    # Real-only case
    print("1. Real amplitudes (H gate):")
    fq.H(wires=[0])(qdev)
    states = torch.view_as_complex(qdev.states)
    print(f"   {states.flatten()}")
    print("   Imaginary parts are all zero")

    # Complex number case
    print("\n2. Complex amplitudes (Phase gate):")
    qdev.reset_states()
    fq.H(wires=[0])(qdev)  # First create superposition
    fq.PHASE(wires=[0])(qdev, params=torch.tensor([0.8]))  # Add phase
    states = torch.view_as_complex(qdev.states)
    print(f"   {states.flatten()}")
    print(f"   |0⟩: {states[0][0].item():.3f}")
    print(f"   |1⟩: {states[0][1].item():.3f}")
    print(f"   Phase of |1⟩: {torch.atan2(states[0][1].imag, states[0][1].real).item():.3f} rad")

    # Physical meaning of phase
    print("\n3. Physical meaning of phase:")
    print("   Global phase of a quantum state is unobservable, but relative phase matters")
    qdev.reset_states()
    fq.H(wires=[0])(qdev)
    fq.PHASE(wires=[0])(qdev, params=torch.tensor([torch.pi]))  # Add π phase
    states = torch.view_as_complex(qdev.states)
    print(f"   After adding π phase: {states.flatten()}")
    print("   Probabilities remain unchanged, but relative phase has changed")

In [11]:
tutorial_05_complex_numbers()


Lesson 0.5: Understanding Complex Amplitudes
1. Real amplitudes (H gate):
   tensor([0.7071+0.j, 0.7071+0.j])
   Imaginary parts are all zero

2. Complex amplitudes (Phase gate):
   tensor([0.7071+0.0000j, 0.4926+0.5072j])
   |0⟩: 0.707+0.000j
   |1⟩: 0.493+0.507j
   Phase of |1⟩: 0.800 rad

3. Physical meaning of phase:
   Global phase of a quantum state is unobservable, but relative phase matters
   After adding π phase: tensor([ 0.7071+0.0000e+00j, -0.7071-6.1817e-08j])
   Probabilities remain unchanged, but relative phase has changed
